# S0 — CMIP6 Historical Catalog Compatibility Analysis / CMIP6历史情景目录兼容性分析

This notebook queries the local CMIP6 historical Dataset-level catalog and answers four questions:

1. **Core member intersection** — which models have `pr` (Amon), `evspsbl` (Amon), and `mrro` (Lmon) on a common member?
2. **P + R member intersection** — which models have `pr` (Amon) and `mrro` (Lmon) on a common member, even without `evspsbl`?
3. **Other variables vs mrro** — for every model, which variables share a common member with `mrro` (Lmon)?
4. **Compatibility checks** — do the passing models have consistent time periods, grids, and fixed fields?

No data is downloaded. No member recommendation or model selection is made — that belongs to a later step.

**中文说明：** 本Notebook查询本地CMIP6 historical Dataset级别目录，回答四个问题：(1) 哪些模型能在同一member上同时提供pr、evspsbl和mrro（完整P–ET–R水量平衡）；(2) 哪些模型能在同一member上同时提供pr和mrro（P→R分析，不要求evspsbl）；(3) 每个模型有哪些变量可以与mrro配对；(4) 通过的模型在时间覆盖、网格和固定场方面是否兼容。本Notebook不下载数据，也不做模型选择或member推荐。

## 1. Configuration & data loading / 配置与数据加载

**中文说明：** 定位caseA目录和ESGF目录文件，定义核心变量组合和分析时段（1985–2014），建立member、时间和网格的索引字典以加速后续查询。

In [18]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display


def locate_case_dir() -> Path:
    for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if d.name == "caseA":
            return d
        nested = d / "case" / "caseA"
        if nested.is_dir():
            return nested
    raise FileNotFoundError("Could not locate case/caseA")


CASE_DIR = locate_case_dir()
CATALOG_JSONL = CASE_DIR / "CMIP_information" / "esgf_historical_model_variable_docs.jsonl"
OUT_DIR = CASE_DIR / "output" / "S0"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CORE = [("pr", "Amon"), ("evspsbl", "Amon"), ("mrro", "Lmon")]

ANALYSIS_START_YEAR = 1985
PASS_STOP = pd.Timestamp("2014-12-01", tz="UTC")
FLAG_STOP = pd.Timestamp("2012-01-01", tz="UTC")

print("Case directory:", CASE_DIR)
print("Raw catalog:", CATALOG_JSONL, "exists=", CATALOG_JSONL.is_file())
print("Output:", OUT_DIR)

Case directory: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA
Raw catalog: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/CMIP_information/esgf_historical_model_variable_docs.jsonl exists= True
Output: /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S0


In [19]:
raw = pd.read_json(CATALOG_JSONL, lines=True)
raw = raw.dropna(subset=["source_id", "variable_id", "table_id", "member_id"])
raw["replica"] = raw["replica"].fillna(False).astype(bool)
raw = raw.sort_values(["master_id", "replica"])
docs = raw.drop_duplicates("master_id", keep="first").copy()


def parse_time(v):
    if v is None or v == "" or (isinstance(v, float) and pd.isna(v)):
        return pd.NaT
    return pd.to_datetime(v, utc=True, errors="coerce")


docs["start"] = docs["datetime_start"].map(parse_time)
docs["stop"] = docs["datetime_stop"].map(parse_time)

ALL_MODELS = sorted(docs["source_id"].unique())

# --- Pre-build dict indexes for fast lookup ---
# member sets: (model, variable, table) -> set of members
member_sets = docs.groupby(["source_id", "variable_id", "table_id"])["member_id"].apply(set).to_dict()

# time index: (model, variable, table, member) -> (earliest_start, latest_stop)
_time_agg = docs.groupby(["source_id", "variable_id", "table_id", "member_id"]).agg(
    start=("start", "min"), stop=("stop", "max")
)
time_index = {k: (row.start, row.stop) for k, row in _time_agg.iterrows()}

# grid index: (model, variable, table, member) -> set of grid_labels
grid_index = (
    docs.dropna(subset=["grid_label"])
    .groupby(["source_id", "variable_id", "table_id", "member_id"])["grid_label"]
    .apply(set)
    .to_dict()
)

print(f"Unique Dataset records: {len(docs):,}")
print(f"Models: {len(ALL_MODELS)}")
print(f"Variables: {docs['variable_id'].nunique()}")
print(f"Index sizes — member_sets: {len(member_sets):,}, time: {len(time_index):,}, grid: {len(grid_index):,}")

Unique Dataset records: 273,607
Models: 75
Variables: 1103
Index sizes — member_sets: 24,969, time: 266,485, grid: 266,485


## 2. Core three-variable member intersection / 核心三变量member交集

For each of the 75 models, find the set of members that have `pr` (Amon), `evspsbl` (Amon), and `mrro` (Lmon). The intersection is the set of members that could run the full P–ET–R pipeline.

**中文说明：** 对75个模型逐一检查：是否存在同一个member同时拥有pr(Amon)、evspsbl(Amon)和mrro(Lmon)三个变量。通过交集的模型可以运行完整的P–ET–R水量平衡分析。

In [20]:
core_rows = []
for model in ALL_MODELS:
    pr_m = member_sets.get((model, "pr", "Amon"), set())
    et_m = member_sets.get((model, "evspsbl", "Amon"), set())
    ro_m = member_sets.get((model, "mrro", "Lmon"), set())
    common = pr_m & et_m & ro_m
    core_rows.append({
        "model": model,
        "pr_Amon_n": len(pr_m),
        "evspsbl_Amon_n": len(et_m),
        "mrro_Lmon_n": len(ro_m),
        "common_members": ", ".join(sorted(common)) if common else "",
        "n_common": len(common),
        "missing": ", ".join(
            [name for name, s in [("pr/Amon", pr_m), ("evspsbl/Amon", et_m), ("mrro/Lmon", ro_m)] if not s]
        ),
    })

core_df = pd.DataFrame(core_rows)
core_pass = core_df[core_df["n_common"] > 0].copy()
core_fail = core_df[core_df["n_common"] == 0].copy()

print(f"Models with common member (core pass): {len(core_pass)}")
print(f"Models without common member (core fail): {len(core_fail)}")
print()
print("=== Core pass ===")
display(core_pass[["model", "pr_Amon_n", "evspsbl_Amon_n", "mrro_Lmon_n", "n_common"]])
print()
print("=== Core fail ===")
display(core_fail[["model", "pr_Amon_n", "evspsbl_Amon_n", "mrro_Lmon_n", "missing"]])

Models with common member (core pass): 65
Models without common member (core fail): 10

=== Core pass ===


,model,pr_Amon_n,evspsbl_Amon_n,mrro_Lmon_n,n_common
0,ACCESS-CM2,10,10,10,10
1,ACCESS-ESM1-5,40,40,40,40
3,AWI-ESM-1-1-LR,1,1,1,1
5,BCC-CSM2-MR,3,3,3,3
6,BCC-ESM1,3,3,3,3
...,...,...,...,...,...
70,NorESM2-MM,3,3,3,3
71,SAM0-UNICON,1,1,1,1
72,TaiESM1,2,2,2,2
73,UKESM1-0-LL,19,19,19,19



=== Core fail ===


,model,pr_Amon_n,evspsbl_Amon_n,mrro_Lmon_n,missing
2,AWI-CM-1-1-MR,5,5,0,mrro/Lmon
4,AWI-ESM-1-REcoM,0,0,1,"pr/Amon, evspsbl/Amon"
13,CIESM,3,0,3,evspsbl/Amon
32,EC-Earth3-ESM-1,0,0,0,"pr/Amon, evspsbl/Amon, mrro/Lmon"
33,EC-Earth3-HR,0,0,0,"pr/Amon, evspsbl/Amon, mrro/Lmon"
36,EC-Earth3P-VHR,0,0,0,"pr/Amon, evspsbl/Amon, mrro/Lmon"
51,IITM-ESM,1,1,0,mrro/Lmon
58,KIOST-ESM,1,0,1,evspsbl/Amon
67,NESM3,5,5,0,mrro/Lmon
68,NorCPM1,30,30,0,mrro/Lmon


## 3. Variables vs mrro member intersection / 变量与mrro的member交集

This section checks which variables share a common member with `mrro` (Lmon). Two levels:

- **3a. P + R** — the specific `pr` (Amon) + `mrro` (Lmon) pair, which is the minimum requirement for P→R spatial relationship analysis. Models that have this pair but lack `evspsbl` can still contribute to P→R.
- **3b. All variables** — every variable in the catalog is checked against `mrro` for member overlap. This identifies which variables could serve as additional predictors or context variables paired with runoff.

**中文说明：** 本节检查哪些变量可以与mrro(Lmon)在同一member上配对。分两个层次：(3a) 专门检查pr+mrro两变量交集——这是P→R空间关系分析的最低要求，即使没有evspsbl也可以做P→R；(3b) 检查目录中所有变量与mrro的member交集，识别哪些变量可以作为额外的输入与径流配对。

In [21]:
mrro_members_by_model = {}
for model in ALL_MODELS:
    mrro_members_by_model[model] = member_sets.get((model, "mrro", "Lmon"), set())

models_with_mrro = [m for m, s in mrro_members_by_model.items() if s]
models_without_mrro = [m for m, s in mrro_members_by_model.items() if not s]
print(f"Models with mrro(Lmon): {len(models_with_mrro)}")
print(f"Models without mrro(Lmon): {len(models_without_mrro)} — {models_without_mrro}")

Models with mrro(Lmon): 68
Models without mrro(Lmon): 7 — ['AWI-CM-1-1-MR', 'EC-Earth3-ESM-1', 'EC-Earth3-HR', 'EC-Earth3P-VHR', 'IITM-ESM', 'NESM3', 'NorCPM1']


### 3a. P + R (pr + mrro) two-variable member intersection / P+R两变量member交集

For each model, intersect the `pr` (Amon) member set with the `mrro` (Lmon) member set. This is the minimum requirement for P→R spatial relationship analysis — no `evspsbl` needed.

**中文说明：** 对每个模型，取pr(Amon)和mrro(Lmon)的member交集。这是P→R空间关系分析的最低要求——不需要evspsbl。有些模型可能有pr+mrro但缺少evspsbl，因此无法做完整P–ET–R水量平衡，但仍可以做P→R分析。

In [22]:
pr_r_rows = []
for model in ALL_MODELS:
    pr_m = member_sets.get((model, "pr", "Amon"), set())
    mrro_m = mrro_members_by_model[model]
    common = pr_m & mrro_m
    et_m = member_sets.get((model, "evspsbl", "Amon"), set())
    in_core = bool(pr_m & et_m & mrro_m)
    pr_r_rows.append({
        "model": model,
        "pr_Amon_n": len(pr_m),
        "mrro_Lmon_n": len(mrro_m),
        "n_common": len(common),
        "common_members": ", ".join(sorted(common)) if common else "",
        "also_in_core_3var": in_core,
        "missing": ", ".join(
            [name for name, s in [("pr/Amon", pr_m), ("mrro/Lmon", mrro_m)] if not s]
        ),
    })

pr_r_df = pd.DataFrame(pr_r_rows)
pr_r_pass = pr_r_df[pr_r_df["n_common"] > 0].copy()
pr_r_fail = pr_r_df[pr_r_df["n_common"] == 0].copy()
pr_r_only = pr_r_pass[~pr_r_pass["also_in_core_3var"]].copy()

print(f"Models with pr + mrro common member: {len(pr_r_pass)}")
print(f"  — also in core P–ET–R (Section 2): {pr_r_pass['also_in_core_3var'].sum()}")
print(f"  — P→R only (have pr+mrro but NOT evspsbl): {len(pr_r_only)}")
print(f"Models without pr + mrro common member: {len(pr_r_fail)}")
print()

if len(pr_r_only) > 0:
    print("=== Models that can run P→R but NOT full P–ET–R ===")
    display(pr_r_only[["model", "pr_Amon_n", "mrro_Lmon_n", "n_common"]])
    print()

print("=== All P + R pass models ===")
display(pr_r_pass[["model", "pr_Amon_n", "mrro_Lmon_n", "n_common", "also_in_core_3var"]])

Models with pr + mrro common member: 67
  — also in core P–ET–R (Section 2): 65
  — P→R only (have pr+mrro but NOT evspsbl): 2
Models without pr + mrro common member: 8

=== Models that can run P→R but NOT full P–ET–R ===


,model,pr_Amon_n,mrro_Lmon_n,n_common
13,CIESM,3,3,3
58,KIOST-ESM,1,1,1



=== All P + R pass models ===


,model,pr_Amon_n,mrro_Lmon_n,n_common,also_in_core_3var
0,ACCESS-CM2,10,10,10,True
1,ACCESS-ESM1-5,40,40,40,True
3,AWI-ESM-1-1-LR,1,1,1,True
5,BCC-CSM2-MR,3,3,3,True
6,BCC-ESM1,3,3,3,True
...,...,...,...,...,...
70,NorESM2-MM,3,3,3,True
71,SAM0-UNICON,1,1,1,True
72,TaiESM1,2,2,2,True
73,UKESM1-0-LL,19,19,19,True


### 3b. All variables vs mrro member intersection / 所有变量与mrro的member交集

For every model with `mrro` (Lmon), check all variables in the catalog for member overlap. This generalizes the P + R check above to all possible variable pairings.

**中文说明：** 对每个拥有mrro(Lmon)的模型，检查目录中所有变量是否与mrro在同一member上配对。这是3a中P+R检查的推广——覆盖所有可能的变量配对，用于识别可以作为驱动因子或辅助变量参与分析的变量。

In [23]:
var_rows = []
for model in ALL_MODELS:
    mrro_m = mrro_members_by_model[model]
    if not mrro_m:
        continue
    model_keys = [(m, v, t) for (m, v, t) in member_sets if m == model and not (v == "mrro" and t == "Lmon")]
    for _, var, table in model_keys:
        common = mrro_m & member_sets[(model, var, table)]
        if common:
            var_rows.append({
                "model": model,
                "variable": var,
                "table_id": table,
                "n_common_with_mrro": len(common),
                "common_members": ", ".join(sorted(common)),
            })

var_mrro_df = pd.DataFrame(var_rows)
print(f"Total variable-model pairs with mrro member overlap: {len(var_mrro_df):,}")
print(f"Unique variables with at least one model overlap: {var_mrro_df['variable'].nunique()}")
display(var_mrro_df.head(10))

Total variable-model pairs with mrro member overlap: 24,371
Unique variables with at least one model overlap: 1103


,model,variable,table_id,n_common_with_mrro,common_members
0,ACCESS-CM2,abs550aer,AERmon,2,"r4i1p1f1, r5i1p1f1"
1,ACCESS-CM2,agessc,Omon,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
2,ACCESS-CM2,areacella,fx,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
3,ACCESS-CM2,areacello,Ofx,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
4,ACCESS-CM2,baresoilFrac,Eyr,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
5,ACCESS-CM2,baresoilFrac,Lmon,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
6,ACCESS-CM2,basin,Ofx,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
7,ACCESS-CM2,bigthetao,Omon,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
8,ACCESS-CM2,bigthetaoga,Omon,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."
9,ACCESS-CM2,c3PftFrac,Lmon,10,"r10i1p1f1, r1i1p1f1, r2i1p1f1, r3i1p1f1, r4i1p..."


In [24]:
var_coverage = (
    var_mrro_df.groupby(["variable", "table_id"])
    .agg(n_models=("model", "nunique"))
    .reset_index()
    .sort_values("n_models", ascending=False)
)
print(f"Variables that can pair with mrro, ranked by how many models support them:")
print(f"(showing top 40 out of {len(var_coverage)})")
print()
display(var_coverage.head(40))

Variables that can pair with mrro, ranked by how many models support them:
(showing top 40 out of 1630)



,variable,table_id,n_models
903,pr,Amon,67
532,hus,Amon,67
1315,ta,Amon,67
1337,tas,Amon,67
1351,tauu,Amon,67
1357,tauv,Amon,67
1029,rlut,Amon,67
1433,tos,Omon,67
1454,ts,Amon,67
667,mrros,Lmon,67


In [25]:
vars_per_model = (
    var_mrro_df.groupby("model")
    .agg(n_variables=("variable", "nunique"))
    .reset_index()
    .sort_values("n_variables", ascending=False)
)
print("Variables pairable with mrro, per model:")
display(vars_per_model)

Variables pairable with mrro, per model:


,model,n_variables
10,CESM2-WACCM,682
11,CESM2-WACCM-FV2,662
8,CESM2,661
9,CESM2-FV2,640
50,IPSL-CM6A-LR,612
...,...,...
27,E3SM-2-1,109
35,FIO-ESM-2-0,90
6,CAMS-CSM1-0,77
3,AWI-ESM-1-REcoM,66


## 4. Time period check / 时间覆盖检查

For models passing Section 2 (core intersection), Section 3a (P + R intersection), and Section 3b (variable-mrro intersection), check whether the relevant records cover the analysis window.

- **Pass**: start year ≤ 1985 and stop ≥ 2014-12
- **Flag**: start year ≤ 1985 and (stop missing or 2012 ≤ stop < 2014-12) — likely a calendar artifact, keep but verify
- **Fail**: otherwise

**中文说明：** 对通过Section 2（核心三变量）、Section 3a（P+R两变量）和Section 3b（其他变量与mrro）的模型，检查相关记录是否覆盖1985–2014分析窗口。Pass表示完整覆盖；Flag表示结束日期缺失或略早于2014-12（可能是日历偏差），保留但需后续验证；Fail表示不满足。

In [26]:
def period_status(start, stop):
    if pd.isna(start) or start.year > ANALYSIS_START_YEAR:
        return "Fail"
    if pd.isna(stop):
        return "Flag"
    if stop >= PASS_STOP:
        return "Pass"
    if stop >= FLAG_STOP:
        return "Flag"
    return "Fail"


def get_period(model, variable, table, member):
    t = time_index.get((model, variable, table, member))
    if t is None:
        return pd.NaT, pd.NaT, "Fail"
    return t[0], t[1], period_status(t[0], t[1])

### 4a. Time period — core variables (Section 2 pass models) / 时间检查——核心变量

For each model with a core common member, check whether **all three** core variables have Pass or Flag on at least one common member.

**中文说明：** 对Section 2通过的模型，检查是否至少有一个common member使得三个核心变量的时间覆盖全部为Pass或Flag。

In [27]:
RANK = {"Pass": 2, "Flag": 1, "Fail": 0}

core_period_rows = []
for _, row in core_pass.iterrows():
    model = row["model"]
    members = [m.strip() for m in row["common_members"].split(",")]
    best_rank = -1
    best_member = None
    best_details = None
    for member in members:
        details = {}
        statuses = []
        for var, tbl in CORE:
            s, e, st = get_period(model, var, tbl, member)
            details[f"{var}_start"] = s
            details[f"{var}_stop"] = e
            details[f"{var}_period"] = st
            statuses.append(st)
        if "Fail" in statuses:
            combined = "Fail"
        elif "Flag" in statuses:
            combined = "Flag"
        else:
            combined = "Pass"
        if RANK[combined] > best_rank:
            best_rank = RANK[combined]
            best_member = member
            best_details = details
            best_status = combined
    core_period_rows.append({
        "model": model,
        "n_common": row["n_common"],
        "example_member": best_member,
        "core_period": best_status,
        **best_details,
    })

core_period_df = pd.DataFrame(core_period_rows)
core_period_pass = core_period_df[core_period_df["core_period"].isin(["Pass", "Flag"])].copy()
core_period_fail = core_period_df[core_period_df["core_period"] == "Fail"].copy()

print(f"Core period Pass: {(core_period_df['core_period']=='Pass').sum()}")
print(f"Core period Flag: {(core_period_df['core_period']=='Flag').sum()}")
print(f"Core period Fail: {(core_period_df['core_period']=='Fail').sum()}")
print()
display(core_period_df[["model", "n_common", "example_member", "core_period",
                         "pr_start", "pr_stop", "evspsbl_start", "evspsbl_stop",
                         "mrro_start", "mrro_stop"]])

Core period Pass: 35
Core period Flag: 28
Core period Fail: 2



,model,n_common,example_member,core_period,pr_start,pr_stop,evspsbl_start,evspsbl_stop,mrro_start,mrro_stop
0,ACCESS-CM2,10,r10i1p1f1,Flag,1850-01-16 00:00:00+00:00,NaT,1850-01-16 00:00:00+00:00,NaT,1850-01-16 00:00:00+00:00,NaT
1,ACCESS-ESM1-5,40,r10i1p1f1,Flag,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,NaT
2,AWI-ESM-1-1-LR,1,r1i1p1f1,Flag,1850-01-16 00:00:00+00:00,NaT,1850-01-16 00:00:00+00:00,NaT,1850-01-16 00:00:00+00:00,NaT
3,BCC-CSM2-MR,3,r1i1p1f1,Pass,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00
4,BCC-ESM1,3,r1i1p1f1,Pass,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00
...,...,...,...,...,...,...,...,...,...,...
60,NorESM2-MM,3,r1i1p1f1,Flag,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1848-10-25 00:00:00+00:00,NaT,1848-10-25 00:00:00+00:00,NaT
61,SAM0-UNICON,1,r1i1p1f1,Pass,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 11:45:00+00:00,2014-12-16 12:00:00+00:00
62,TaiESM1,2,r1i1p1f1,Pass,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00,1850-01-16 12:00:00+00:00,2014-12-16 12:00:00+00:00
63,UKESM1-0-LL,19,r4i1p1f2,Pass,1850-01-16 00:00:00+00:00,2014-12-16 00:00:00+00:00,1850-01-16 00:00:00+00:00,2014-12-16 00:00:00+00:00,1850-01-16 00:00:00+00:00,2014-12-16 00:00:00+00:00


### 4b. Time period — all variables vs mrro (Section 3 pairs) / 时间检查——所有变量与mrro

For each variable-model pair from Section 3 (including pr), check whether **both** the variable and mrro have Pass or Flag on at least one of their common members. P + R time check results are included here — filter for `variable == "pr"` and `table_id == "Amon"` to extract them.

**中文说明：** 对Section 3中每一对变量-模型组合（包括pr），检查是否至少有一个common member使得该变量和mrro的时间覆盖同时为Pass或Flag。P+R的时间检查结果也包含在内——过滤`variable=="pr"`和`table_id=="Amon"`即可提取。

In [28]:
var_period_rows = []
for _, row in var_mrro_df.iterrows():
    model = row["model"]
    var = row["variable"]
    tbl = row["table_id"]
    members = [m.strip() for m in row["common_members"].split(",")]
    best_combined = "Fail"
    best_member = members[0]
    best_var_period = "Fail"
    best_mrro_period = "Fail"
    for member in members:
        _, _, var_st = get_period(model, var, tbl, member)
        _, _, mrro_st = get_period(model, "mrro", "Lmon", member)
        if "Fail" in (var_st, mrro_st):
            combined = "Fail"
        elif "Flag" in (var_st, mrro_st):
            combined = "Flag"
        else:
            combined = "Pass"
        if RANK[combined] > RANK[best_combined]:
            best_combined = combined
            best_member = member
            best_var_period = var_st
            best_mrro_period = mrro_st
    var_period_rows.append({
        "model": model,
        "variable": var,
        "table_id": tbl,
        "n_common_with_mrro": row["n_common_with_mrro"],
        "example_member": best_member,
        "var_period": best_var_period,
        "mrro_period": best_mrro_period,
        "combined_period": best_combined,
    })

var_period_df = pd.DataFrame(var_period_rows)
var_period_pass = var_period_df[var_period_df["combined_period"].isin(["Pass", "Flag"])].copy()
var_period_fail = var_period_df[var_period_df["combined_period"] == "Fail"].copy()

print(f"Variable-model pairs passing time check: {len(var_period_pass)} (Pass: {(var_period_df['combined_period']=='Pass').sum()}, Flag: {(var_period_df['combined_period']=='Flag').sum()})")
print(f"Variable-model pairs failing time check: {len(var_period_fail)}")
print()
var_period_coverage = (
    var_period_pass.groupby(["variable", "table_id"])
    .agg(n_models=("model", "nunique"))
    .reset_index()
    .sort_values("n_models", ascending=False)
)
print("Top 40 variables passing time check, by model count:")
display(var_period_coverage.head(40))

Variable-model pairs passing time check: 23425 (Pass: 12719, Flag: 10706)
Variable-model pairs failing time check: 946

Top 40 variables passing time check, by model count:


,variable,table_id,n_models
655,mrros,Lmon,66
1482,va,Amon,66
1289,ta,Amon,66
523,hus,Amon,66
1449,ua,Amon,66
1013,rlut,Amon,65
1331,tauv,Amon,65
1565,zg,Amon,65
1325,tauu,Amon,65
1405,tos,Omon,65


## 5. Grid check / 网格一致性检查

For models/pairs passing the time check, verify whether the grid labels are consistent.

**中文说明：** 对通过时间检查的模型和变量对，验证网格标签是否一致。如果变量与mrro使用不同网格（如gn vs gr），则需要重网格化才能配对分析。

### 5a. Grid — core variables / 网格检查——核心变量

**中文说明：** 检查Section 2通过时间检查的模型中，pr、evspsbl和mrro三个变量是否使用相同网格。

In [29]:
core_grid_rows = []
for _, row in core_period_pass.iterrows():
    model = row["model"]
    member = row["example_member"]
    all_grids = {}
    for var, tbl in CORE:
        g = grid_index.get((model, var, tbl, member), set())
        all_grids[f"{var}_grid"] = ", ".join(sorted(g)) if g else ""
    unique_grids = set()
    for v in all_grids.values():
        if v:
            unique_grids.update(v.split(", "))
    core_grid_rows.append({
        "model": model,
        "example_member": member,
        **all_grids,
        "same_grid": len(unique_grids) <= 1,
        "grids": ", ".join(sorted(unique_grids)),
    })

core_grid_df = pd.DataFrame(core_grid_rows)
print(f"Same grid across core variables: {core_grid_df['same_grid'].sum()}")
print(f"Different grids (need regrid): {(~core_grid_df['same_grid']).sum()}")
print()
display(core_grid_df)

Same grid across core variables: 62
Different grids (need regrid): 1



,model,example_member,pr_grid,evspsbl_grid,mrro_grid,same_grid,grids
0,ACCESS-CM2,r10i1p1f1,gn,gn,gn,True,gn
1,ACCESS-ESM1-5,r10i1p1f1,gn,gn,gn,True,gn
2,AWI-ESM-1-1-LR,r1i1p1f1,gn,gn,gn,True,gn
3,BCC-CSM2-MR,r1i1p1f1,gn,gn,gn,True,gn
4,BCC-ESM1,r1i1p1f1,gn,gn,gn,True,gn
...,...,...,...,...,...,...,...
58,NorESM2-MM,r1i1p1f1,gn,gn,gn,True,gn
59,SAM0-UNICON,r1i1p1f1,gn,gn,gn,True,gn
60,TaiESM1,r1i1p1f1,gn,gn,gn,True,gn
61,UKESM1-0-LL,r4i1p1f2,gn,gn,gn,True,gn


### 5b. Grid — all variables vs mrro / 网格检查——所有变量与mrro

P + R grid check results are included here — filter for `variable == "pr"` and `table_id == "Amon"` to extract them.

**中文说明：** 检查Section 3中通过时间检查的所有变量-模型对（包括pr），变量与mrro是否使用相同网格。P+R的网格检查结果也包含在内——过滤`variable=="pr"`和`table_id=="Amon"`即可提取。

In [30]:
var_grid_rows = []
for _, row in var_period_pass.iterrows():
    model = row["model"]
    var = row["variable"]
    tbl = row["table_id"]
    member = row["example_member"]
    var_g = grid_index.get((model, var, tbl, member), set())
    mrro_g = grid_index.get((model, "mrro", "Lmon", member), set())
    var_grid_rows.append({
        "model": model,
        "variable": var,
        "table_id": tbl,
        "var_grid": ", ".join(sorted(var_g)),
        "mrro_grid": ", ".join(sorted(mrro_g)),
        "same_grid": bool(var_g & mrro_g),
    })

var_grid_df = pd.DataFrame(var_grid_rows)
print(f"Same grid: {var_grid_df['same_grid'].sum()}")
print(f"Different grid (need regrid): {(~var_grid_df['same_grid']).sum()}")
if (~var_grid_df["same_grid"]).any():
    print()
    print("Pairs needing regrid:")
    display(var_grid_df[~var_grid_df["same_grid"]])

Same grid: 20360
Different grid (need regrid): 3065

Pairs needing regrid:


,model,variable,table_id,var_grid,mrro_grid,same_grid
4955,CIESM,hfbasin,Omon,gn,gr,False
4956,CIESM,hfds,Omon,gn,gr,False
4965,CIESM,msftbarot,Omon,gn,gr,False
4966,CIESM,msftyz,Omon,gn,gr,False
4985,CIESM,siage,SImon,gn,gr,False
...,...,...,...,...,...,...
23385,UKESM1-1-LL,siconca,SImon,gr,gn,False
23394,UKESM1-1-LL,ta,AERmonZ,gnz,gn,False
23404,UKESM1-1-LL,thetaoga,Omon,gm,gn,False
23419,UKESM1-1-LL,volo,Omon,gm,gn,False


## 6. Fixed field check / 固定场可用性检查

Check availability of `sftlf` (fx), `areacella` (fx), and optionally `sftgif` (fx or LImon). Fixed fields are not member-specific — any member having them is enough.

**中文说明：** 检查陆地面积比例`sftlf`、网格面积`areacella`和陆地冰比例`sftgif`是否可用。固定场不区分member，只要模型有任意一个member提供即可。`sftlf`和`areacella`是必需的（用于区分海洋与陆地、计算面积权重），`sftgif`是可选的（用于区分冰盖与非冰陆地）。

In [31]:
FIXED_FIELDS = [("sftlf", "fx"), ("areacella", "fx"), ("sftgif", "fx"), ("sftgif", "LImon")]

fixed_rows = []
for model in ALL_MODELS:
    row = {"model": model}
    for var, tbl in FIXED_FIELDS:
        row[f"{var}_{tbl}"] = (model, var, tbl) in member_sets
    row["sftgif_any"] = row.get("sftgif_fx", False) or row.get("sftgif_LImon", False)
    fixed_rows.append(row)

fixed_df = pd.DataFrame(fixed_rows)
print(f"sftlf (fx): {fixed_df['sftlf_fx'].sum()} / {len(fixed_df)} models")
print(f"areacella (fx): {fixed_df['areacella_fx'].sum()} / {len(fixed_df)} models")
print(f"sftgif (fx or LImon): {fixed_df['sftgif_any'].sum()} / {len(fixed_df)} models")
print()
display(fixed_df)

sftlf (fx): 49 / 75 models
areacella (fx): 50 / 75 models
sftgif (fx or LImon): 36 / 75 models



,model,sftlf_fx,areacella_fx,sftgif_fx,sftgif_LImon,sftgif_any
0,ACCESS-CM2,True,True,True,True,True
1,ACCESS-ESM1-5,True,True,True,True,True
2,AWI-CM-1-1-MR,True,True,False,False,False
3,AWI-ESM-1-1-LR,True,True,True,False,True
4,AWI-ESM-1-REcoM,False,False,False,False,False
...,...,...,...,...,...,...
70,NorESM2-MM,True,True,False,False,False
71,SAM0-UNICON,True,True,True,False,True
72,TaiESM1,True,True,False,False,False
73,UKESM1-0-LL,False,False,False,False,False


## 7. Summary / 总结

Combine results from Sections 2–6 into a single overview. No member recommendation or model selection is made here.

Three compatibility funnels are reported:
- **Core P–ET–R** — models passing the full three-variable pipeline (Section 2)
- **P + R** — models that can run P→R analysis, extracted from the general variable-mrro checks (Sections 3b → 4b → 5b)
- **Variable × mrro matrix** — all variables ranked by how many models pass member + time + grid

**中文说明：** 将Section 2–6的结果合并为三条兼容性漏斗：(1) 核心P–ET–R三变量流水线；(2) P+R两变量分析（从通用变量-mrro检查中提取pr的行）；(3) 所有变量按通过member+时间+网格检查的模型数排名。本节不做模型选择或member推荐。

In [32]:
summary_core = core_df[["model", "n_common", "missing"]].merge(
    core_period_df[["model", "core_period"]].rename(columns={"core_period": "time_status"}),
    on="model", how="left",
).merge(
    core_grid_df[["model", "same_grid", "grids"]],
    on="model", how="left",
).merge(
    fixed_df[["model", "sftlf_fx", "areacella_fx", "sftgif_any"]],
    on="model", how="left",
)
summary_core["time_status"] = summary_core["time_status"].fillna("—")

print("=== Core P–ET–R compatibility summary (all 75 models) ===")
print(f"  Core member intersection: {(summary_core['n_common'] > 0).sum()} pass")
print(f"  + time check: {summary_core['time_status'].isin(['Pass','Flag']).sum()} pass")
print(f"  + same grid: {(summary_core['same_grid'] == True).sum()} pass")
print(f"  + sftlf: {((summary_core['same_grid'] == True) & (summary_core['sftlf_fx'] == True)).sum()} pass")
print(f"  + areacella: {((summary_core['same_grid'] == True) & (summary_core['areacella_fx'] == True)).sum()} pass")
print()
display(summary_core)

# --- P + R compatibility summary ---
print()
print("=== P + R (pr + mrro) compatibility summary ===")
pr_member_n = len(pr_r_pass)
pr_time = var_period_pass.loc[
    (var_period_pass["variable"] == "pr") & (var_period_pass["table_id"] == "Amon")
]
pr_time_n = pr_time["model"].nunique()
pr_time_models = set(pr_time["model"])
pr_grid = var_grid_df.loc[
    (var_grid_df["variable"] == "pr") & (var_grid_df["table_id"] == "Amon") & var_grid_df["same_grid"]
]
pr_grid_n = pr_grid["model"].nunique()
pr_grid_models = set(pr_grid["model"])
pr_sftlf_n = fixed_df.loc[fixed_df["model"].isin(pr_grid_models) & fixed_df["sftlf_fx"]].shape[0]
pr_areacella_n = fixed_df.loc[fixed_df["model"].isin(pr_grid_models) & fixed_df["areacella_fx"]].shape[0]

print(f"  P+R member intersection: {pr_member_n} pass")
print(f"  + time check: {pr_time_n} pass")
print(f"  + same grid: {pr_grid_n} pass")
print(f"  + sftlf: {pr_sftlf_n} pass")
print(f"  + areacella: {pr_areacella_n} pass")

pr_only_models = set(pr_r_pass["model"]) - set(core_pass["model"])
core_only_models = set(core_pass["model"]) - set(pr_r_pass["model"])
if pr_only_models:
    print(f"\n  Models in P+R but NOT in core P–ET–R: {sorted(pr_only_models)}")
if core_only_models:
    print(f"\n  Models in core P–ET–R but NOT in P+R: {sorted(core_only_models)}")
if not pr_only_models and not core_only_models:
    print(f"\n  P+R and core P–ET–R have the same model set")

=== Core P–ET–R compatibility summary (all 75 models) ===
  Core member intersection: 65 pass
  + time check: 63 pass
  + same grid: 62 pass
  + sftlf: 43 pass
  + areacella: 45 pass



,model,n_common,missing,time_status,same_grid,grids,sftlf_fx,areacella_fx,sftgif_any
0,ACCESS-CM2,10,,Flag,True,gn,True,True,True
1,ACCESS-ESM1-5,40,,Flag,True,gn,True,True,True
2,AWI-CM-1-1-MR,0,mrro/Lmon,—,NaN,NaN,True,True,False
3,AWI-ESM-1-1-LR,1,,Flag,True,gn,True,True,True
4,AWI-ESM-1-REcoM,0,"pr/Amon, evspsbl/Amon",—,NaN,NaN,False,False,False
...,...,...,...,...,...,...,...,...,...
70,NorESM2-MM,3,,Flag,True,gn,True,True,False
71,SAM0-UNICON,1,,Pass,True,gn,True,True,True
72,TaiESM1,2,,Pass,True,gn,True,True,False
73,UKESM1-0-LL,19,,Pass,True,gn,False,False,False



=== P + R (pr + mrro) compatibility summary ===
  P+R member intersection: 67 pass
  + time check: 65 pass
  + same grid: 64 pass
  + sftlf: 43 pass
  + areacella: 45 pass

  Models in P+R but NOT in core P–ET–R: ['CIESM', 'KIOST-ESM']


In [33]:
var_summary = var_mrro_df.groupby(["variable", "table_id"]).agg(
    n_models_member=("model", "nunique"),
).reset_index()

var_time_summary = var_period_pass.groupby(["variable", "table_id"]).agg(
    n_models_time=("model", "nunique"),
).reset_index()

var_grid_pass = var_grid_df[var_grid_df["same_grid"]]
var_grid_summary = var_grid_pass.groupby(["variable", "table_id"]).agg(
    n_models_grid=("model", "nunique"),
).reset_index()

summary_var = var_summary.merge(
    var_time_summary, on=["variable", "table_id"], how="left",
).merge(
    var_grid_summary, on=["variable", "table_id"], how="left",
)
summary_var["n_models_time"] = summary_var["n_models_time"].fillna(0).astype(int)
summary_var["n_models_grid"] = summary_var["n_models_grid"].fillna(0).astype(int)
summary_var = summary_var.sort_values("n_models_grid", ascending=False)

print("=== Variable vs mrro compatibility summary ===")
print(f"Variables with member overlap: {len(summary_var)}")
print(f"Variables passing member + time + grid on ≥10 models: {(summary_var['n_models_grid'] >= 10).sum()}")
print()
print("Top 50 variables (by models passing all checks):")
display(summary_var.head(50))

=== Variable vs mrro compatibility summary ===
Variables with member overlap: 1630
Variables passing member + time + grid on ≥10 models: 723

Top 50 variables (by models passing all checks):


,variable,table_id,n_models_member,n_models_time,n_models_grid
667,mrros,Lmon,67,66,66
1477,ua,Amon,67,66,65
1510,va,Amon,67,66,65
1315,ta,Amon,67,66,65
532,hus,Amon,67,66,65
672,mrso,Lmon,65,64,64
1351,tauu,Amon,67,65,64
952,psl,Amon,67,65,64
942,ps,Amon,67,65,64
903,pr,Amon,67,65,64


In [34]:
summary_core.to_csv(OUT_DIR / "S0_core_compatibility.csv", index=False)
summary_var.to_csv(OUT_DIR / "S0_variable_mrro_compatibility.csv", index=False)
var_mrro_df.to_csv(OUT_DIR / "S0_variable_mrro_member_detail.csv", index=False)
pr_r_df.to_csv(OUT_DIR / "S0_pr_mrro_compatibility.csv", index=False)

print("Saved to", OUT_DIR)
for f in sorted(OUT_DIR.glob("S0_*.csv")):
    print(" ", f.name)

Saved to /Users/mimi/Documents/Code/Github/GSA-work/explore/sythetic/case/caseA/output/S0
  S0_core_compatibility.csv
  S0_pr_mrro_compatibility.csv
  S0_variable_mrro_compatibility.csv
  S0_variable_mrro_member_detail.csv
